In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import learning_curve

In [2]:
#使わなくなったやつ
#import pickle
#import lightgbm as lgb
#import xgboost as xgb

#from sklearn.model_selection import train_test_split
#from sklearn.model_selection import KFold, cross_val_score
#from catboost import CatBoostRegressor
#from sklearn.model_selection import GridSearchCV

In [3]:
#前処理したデータ
client_data = pd.read_csv("client_data.csv")
pre_data = pd.read_csv("pre_data.csv")

In [4]:
#Xとyに分割
client_X = client_data.drop(columns=["y"])
client_y = client_data["y"]

X = pre_data.drop(columns=["y"])
y = pre_data["y"]

In [5]:
# KFoldの設定
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [6]:
# XGBRegressorのパラメータ設定
xgb_params = {
    'n_estimators': 500,
    'max_depth': 5,
    'learning_rate': 0.1,
    'random_state': 42
}

In [7]:
# クロスバリデーションの準備
client_models = []
client_predictions = np.zeros(client_X.shape[0])

models = []
predictions = np.zeros(X.shape[0])

In [8]:
# クロスバリデーションでモデルを学習し、予測値を計算(クライアント)
for client_train_index, client_val_index in kf.split(client_X):
    client_X_train, client_X_val = client_X.iloc[client_train_index], client_X.iloc[client_val_index]
    client_y_train, client_y_val = client_y.iloc[client_train_index], client_y.iloc[client_val_index]
    
    client_model = XGBRegressor(**xgb_params)
    client_model.fit(client_X_train, client_y_train)
    
    # 各分割のモデルをリストに保存
    client_models.append(client_model)
    
    # 各分割の予測値を集計
    client_predictions[client_val_index] = client_model.predict(client_X_val)

In [9]:
# クロスバリデーションでモデルを学習し、予測値を計算
for train_index, val_index in kf.split(X):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    model = XGBRegressor(**xgb_params)
    model.fit(X_train, y_train)
    
    # 各分割のモデルをリストに保存
    models.append(model)
    
    # 各分割の予測値を集計
    predictions[val_index] = model.predict(X_val)

In [10]:
# 全体の予測値を計算（各分割の予測値の平均） ・・・クライアント
client_final_predictions = np.zeros(client_X.shape[0])
for client_model in client_models:
    client_final_predictions += client_model.predict(client_X)

client_final_predictions /= len(client_models)

In [11]:
# 全体の予測値を計算（各分割の予測値の平均）
final_predictions = np.zeros(X.shape[0])
for model in models:
    final_predictions += model.predict(X)

final_predictions /= len(models)

In [12]:
# モデルの性能評価(クライアント)
client_mae = mean_absolute_error(client_y, client_final_predictions)
print(f'Mean Absolute Error(client): {client_mae}')

Mean Absolute Error(client): 0.005103503454299543


In [13]:
# モデルの性能評価
mae = mean_absolute_error(y, final_predictions)
print(f'Mean Absolute Error: {mae}')

Mean Absolute Error: 0.53742063004745


In [14]:
# モデルを保存
joblib.dump(client_models, "Apple_client_model.pkl")
joblib.dump(models, "Apple_model.pkl")

['Apple_model.pkl']